# Business entity resolution — procedure

Same steps as `src/rank_match.py`. Do not run the full training or test commands while `output_v3` scoring is still writing. This notebook is the procedure in order. It does not reload the 10 million Source 2 and Source 3 rows.

Score in use: macro F0.5. Precision counts twice as much as recall. Country is an open string (US, India, and France on the test set). No external lookup.

## Step 1. Read the TSV files with a tab separator

Columns are `entity_id`, `business_name`, `business_address`, `country`. Ground truth is `source1_entity_id` and a comma-separated `matched_entity_ids` (empty means no match). Source is the `S1-` / `S2-` / `S3-` prefix, not a separate column.

## Step 2. Normalize text before any comparison

Lower case, turn `&` into `and`, expand then strip legal endings (`Ltd`, `Pvt`, `Inc`, `Corp`, `LLC`), and expand address shorts (`Rd`, `St`, `Ave`). There is no dictionary such as `LLT = company` or `TM = company`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))
from rank_match import block_keys
from ber.normalize import normalize_name, normalize_address

print(normalize_name("Sai Trading Private Limited"))
print(normalize_address("529 Tallmadge Rd, Kent"))
print(block_keys("My Builders", "1A Nazar Ali Lane, Kolkata, West Bengal", "India"))

## Step 3. Batch before search

Each record emits a few keys, and only records that share a key are compared. A Source 1 row is never checked against every Source 2 and Source 3 row.

Keys, all prefixed by the country string:

1. First two significant name words, sorted (and the first-and-third pair when a third word exists).
2. One PIN or ZIP of 5 or more digits.
3. The two longest distinctive address words.
4. One locality word of 8 or more letters.
5. House number plus that locality word.

A key seen on more than 25 businesses is dropped, so a common name cannot pull in every city. Each Source 1 row keeps at most 12 candidates. Address and house-number keys outrank a name-only key.

This index is built in `rank_match.py` (`build_block_db`) before any classifier score is computed.

## Step 4. Score only those candidates

`HistGradientBoostingClassifier` (scikit-learn) is trained on the training ground truth. Features for a pair:

- name closeness: max of token-sort ratio and plain ratio, after normalization
- address closeness with words sorted
- how far one address's words are contained in the other
- whether either address is empty

A candidate is kept when the predicted probability is at least 0.72. That cutoff won on held-out Source 1 businesses. Same address can match even when the names differ (trade name, domain, other script). A close name with a clearly different address does not match.

Held-out F0.5 for this model was 0.678. A hand rule on the same candidates scored 0.582. Earlier portal files, which used only a weighted string score, scored 0.419 and 0.416. An earlier local 0.89 figure injected the true match into the pool and is not a leaderboard score.

In [ ]:
# Full data commands. Do not start these while output_v3 scoring is running.
#
# Held-out check (writes artifacts/train_blocks.sqlite and artifacts/rule.json):
#   python src/rank_match.py --mode eval --stride 400 --max-cands 12
#
# Test file (writes output_v3/matching_results.tsv and candidate_pairs.tsv):
#   python src/rank_match.py --mode infer --max-cands 12 --output-dir output_v3
#
# Validate, from the student_resource directory, only after infer prints DONE:
#   python3 utils/validate_submission.py \
#       --matching output_v3/matching_results.tsv \
#       --candidate output_v3/candidate_pairs.tsv \
#       --test-dir dataset/test
print("Procedure only. Full commands are in the comments above.")

## What would raise the score

Blocking recovered about 65% of true training pairs. Pairs that never enter the list of 12 cannot be saved by the model. The useful changes are typo-tolerant keys that are still deleted when too common, a small abbreviation map (`LLT`, `TM`, and similar), and training on more than the 5,500 Source 1 rows used for the held-out check. Keep the candidate list small: the final review prefers fewer candidates per Source 1 business. Do not upload a partial `output_v3` file.